## Busqueda de hiperparamtros (Semana 7) ##

In [7]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, Callback
import matplotlib.pyplot as plt
import os, logging
from tabulate import tabulate

import joblib
import warnings
# === NUEVA IMPORTACIÓN PARA BÚSQUEDA DE HIPERPARÁMETROS ===
import optuna
from optuna.integration import TFKerasPruningCallback # Para Early Stopping dentro de Optuna
# =========================================================

# Logging mínimo
logging.basicConfig(
    filename="../LOGS/10-entrenamiento_lstm.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)



In [8]:
# =========================================================
# === 1. CONFIGURACIÓN INICIAL DE TENSORFLOW Y GLOBALES ===
# =========================================================

warnings.filterwarnings("ignore")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Configuración de GPU (si aplica)
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ Se detectaron y configuraron {len(gpus)} GPU(s).")
    except RuntimeError as e:
        print(f"Error al configurar GPU: {e}")
else:
    print("ℹ️ Ejecutando en CPU.")


# --- VARIABLES GLOBALES ---
FILE_PATH = "../DATA/PROCESSED/PROCESSED_GL_DATA_ALL_P_L_ANNUAL_MONTHLY_PER_ENTERPRISEV6.csv"
LOOK_BACK = 12
EPOCHS = 1000
# LEARNING_RATE = 0.0005 # Se convierte en un hiperparámetro a buscar
FUTURE_MONTHS = 12
PATIENCE_EARLY_STOPPING = 5
N_SPLITS = 5 # Se mantiene el valor, aunque no se usa en el entrenamiento por entidad
MIN_DATA_POINTS = LOOK_BACK * 2 + 1

OUTPUT_DIR = "../REPORTS/RESULT_LSTM"
MODELS_DIR = "../MODELS/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# === NUEVAS VARIABLES GLOBALES PARA OPTUNA (PRESUPUESTO) ===
N_TRIALS_OPTUNA = 15 # Presupuesto: Número de experimentos a probar (ej. 15)
TIMEOUT_OPTUNA = 3600 # Presupuesto: Tiempo máximo en segundos (ej. 1 hora)
BEST_HYPERPARAMS = {} # Almacenará los mejores resultados de la búsqueda
# ==========================================================

# Logging
logging.basicConfig(
    filename="../LOGS/10-entrenamiento_lstm.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logging.info("1. INICIO DE CONFIGURACIÓN Y CARGA DE DATOS GLOBAL")

# ----------------------------------------------------
# --- CARGA Y PREPROCESAMIENTO GLOBAL ---
# ----------------------------------------------------

try:
    df_full = pd.read_csv(FILE_PATH)
except FileNotFoundError:
    print(f"ERROR: No se encontró el archivo {FILE_PATH}.")
    print("Por favor, asegúrate de que el archivo esté en la ruta correcta.")
    exit()

ENTERPRISE_ID_TO_PROCESS = 0
df_full = df_full[df_full['PORTFOLIOID'] > ENTERPRISE_ID_TO_PROCESS].copy()
df_full['NETINCOME'] = pd.to_numeric(df_full['NETINCOME'], errors='coerce')
df_full = df_full[(df_full['MONTHID'] >= 1) & (df_full['MONTHID'] <= 12)].copy()
df_full.dropna(subset=['NETINCOME', 'YEARID', 'MONTHID', 'PORTFOLIOID', 'PORTFOLIONAME'], inplace=True)
df_full['DATE'] = pd.to_datetime(df_full['YEARID'].astype(str) + '-' + df_full['MONTHID'].astype(str) + '-01')

portfolio_ids = df_full['PORTFOLIOID'].unique()
if len(portfolio_ids) == 0:
    print("ERROR: El DataFrame no contiene PORTFOLIOID válidos después de la limpieza y el filtrado.")
    exit()

print(f"Total de portafolios a analizar: {len(portfolio_ids)}")
print("-" * 50)

all_portfolio_results = []


ℹ️ Ejecutando en CPU.
Total de portafolios a analizar: 4
--------------------------------------------------


In [27]:
# =========================================================
# === 2. FUNCIONES AUXILIARES Y DE MODELADO ===
# =========================================================

class EpochLogger(Callback):
    """Callback personalizado para mostrar el inicio y el final de cada época."""
    def on_epoch_begin(self, epoch, logs=None):
        total_epochs = self.params.get('epochs', EPOCHS)
        if (epoch + 1) % 10 == 1 or epoch == 0:
            print(f"\n[🚀] INICIO DE LA ÉPOCA {epoch + 1}/{total_epochs}")

    def on_epoch_end(self, epoch, logs=None):
        loss = logs.get('loss', 'N/A')
        total_epochs = self.params.get('epochs', EPOCHS)
        if (epoch + 1) % 10 == 0 or (epoch + 1) == total_epochs:
            print(f"[✅] FIN DE LA ÉPOCA {epoch + 1}/{total_epochs} | Pérdida (Loss): {loss:.6f}")


def create_dataset(dataset, look_back=1):
    """Crea las secuencias de entrada (X) y salida (Y) para el modelo LSTM."""
    X, Y = [], []
    for i in range(len(dataset) - look_back):
        X.append(dataset[i:(i + look_back), 0])
        Y.append(dataset[i + look_back, 0])
    return np.array(X), np.array(Y)

def calculate_directional_accuracy(y_true, y_pred):
    """
    Calcula la Precisión Direccional (DA) - cuán a menudo la predicción acierta
    la dirección del cambio (subida o bajada) en comparación con el valor anterior.
    """
    # Asegurarse de que las entradas son 1D arrays
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()
    
    # Se calcula la dirección del cambio: 1 si sube o se mantiene, -1 si baja.
    # La predicción (y_pred[1:]) se compara con el valor REAL anterior (y_true[:-1])
    y_true_direction = np.sign(y_true[1:] - y_true[:-1])
    y_pred_direction = np.sign(y_pred[1:] - y_true[:-1])
    
    # Calcula cuántas veces la dirección predicha coincide con la dirección real
    correct_directions = np.sum(y_true_direction == y_pred_direction)
    total_directions = len(y_true_direction)
    
    if total_directions == 0:
        return np.nan
        
    return correct_directions / total_directions


def calculate_metrics_for_model(model, X_test, y_test, scaler):
    """ Función auxiliar para calcular métricas para un modelo dado. """
    X_test_reshaped = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))
    test_predict_scaled = model.predict(X_test_reshaped, verbose=0)

    test_predict = scaler.inverse_transform(test_predict_scaled)
    y_test_original = scaler.inverse_transform(y_test.reshape(-1, 1))

    rmse = np.sqrt(mean_squared_error(y_test_original, test_predict))
    mae = mean_absolute_error(y_test_original, test_predict)
    r2 = r2_score(y_test_original, test_predict)
    
    # Métrica de 'Accuracy' para regresión: Precisión Direccional (DA)
    da = calculate_directional_accuracy(y_test_original, test_predict)
    
    return rmse, mae, r2, da

# ------------------------------------------------------------------
# --- FUNCIÓN DE CONSTRUCCIÓN UNIFICADA (Para Optuna) ---
# ------------------------------------------------------------------

def build_lstm_model(look_back, n_units_l1, n_units_l2, dropout_rate, learning_rate, optimizer_name='Adam'):
    """Define y compila el modelo LSTM con hiperparámetros variables."""
    model = Sequential([
        Input(shape=(look_back, 1)),
        LSTM(n_units_l1, return_sequences=True),
        Dropout(dropout_rate),
        LSTM(n_units_l2),
        Dropout(dropout_rate),
        Dense(1, activation='linear')
    ])
    
    if optimizer_name == 'Adam':
        optimizer = Adam(learning_rate=learning_rate)
    elif optimizer_name == 'RMSprop':
        optimizer = tf.keras.optimizers.RMSprop(learning_rate=learning_rate)
    else: # Default
        optimizer = Adam(learning_rate=learning_rate)
        
    model.compile(optimizer=optimizer, loss='mean_squared_error', metrics=['mae'])
    return model

# ------------------------------------------------------------------
# --- FUNCIÓN OBJETIVO PARA OPTUNA (Definir Espacio y Presupuesto) ---
# ------------------------------------------------------------------

# Se usa un subconjunto del set agregado para que la búsqueda sea más rápida (limitación de presupuesto)
def objective(trial, X_train, y_train, X_val, y_val, look_back):
    
    # 1. Definir Espacio de Búsqueda (Random/Bayes)
    # Optuna usa Samplers configurables (TPE para Bayes, RandomSampler para Random)
    n_units_l1 = trial.suggest_categorical('n_units_l1', [64, 128, 256])
    n_units_l2 = trial.suggest_categorical('n_units_l2', [32, 64, 128])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.4)
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True)
    optimizer_name = trial.suggest_categorical('optimizer_name', ['Adam', 'RMSprop'])
    
    # 2. Construir el modelo
    model = build_lstm_model(
        look_back=look_back,
        n_units_l1=n_units_l1,
        n_units_l2=n_units_l2,
        dropout_rate=dropout_rate,
        learning_rate=learning_rate,
        optimizer_name=optimizer_name
    )
    
    # 3. Early Stopping con Pruning (Parada temprana y poda de experimentos malos)
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=PATIENCE_EARLY_STOPPING, verbose=0, restore_best_weights=True),
        TFKerasPruningCallback(trial, monitor='val_loss') # Poda de Optuna
    ]
    
    X_train_reshaped = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
    validation_tuple = (np.reshape(X_val, (X_val.shape[0], X_val.shape[1], 1)), y_val)
    
    # 4. Entrenamiento
    history = model.fit(
        X_train_reshaped,
        y_train,
        epochs=EPOCHS,
        batch_size=1,
        verbose=0,
        validation_data=validation_tuple,
        callbacks=callbacks
    )
    
    # 5. Métrica a optimizar (Objetivo: Minimizar la pérdida de validación)
    best_val_loss = min(history.history['val_loss'])
    return best_val_loss


# ------------------------------------------------------------------
# --- FUNCIÓN DE BÚSQUEDA DE HIPERPARÁMETROS ---
# ------------------------------------------------------------------

def find_optimal_hyperparameters(df_full):
    """Realiza la búsqueda de hiperparámetros con Optuna usando un portafolio de muestra."""
    print("\n" + "="*80)
    print(f"=== ⚙️ INICIO BÚSQUEDA DE HIPERPARÁMETROS (OPTUNA: Random Search/TPE) ===")
    print(f"Presupuesto: {N_TRIALS_OPTUNA} pruebas o {TIMEOUT_OPTUNA/60} minutos.")
    print("==================================================================================")
    
    # Usar el primer portafolio con datos suficientes como set de validación global
    df_sample = None
    for p_id in portfolio_ids:
        df_group = df_full[df_full['PORTFOLIOID'] == p_id].copy()
        data_full = df_group.groupby('DATE')['NETINCOME'].sum().values.reshape(-1, 1)
        if len(data_full) >= MIN_DATA_POINTS:
            df_sample = df_group.groupby('DATE')['NETINCOME'].sum().reset_index()
            break
            
    if df_sample is None:
        print("ADVERTENCIA: No hay portafolio con datos suficientes para Optuna. Usando valores por defecto.")
        return {'n_units_l1': 256, 'n_units_l2': 128, 'dropout_rate': 0.3, 'learning_rate': 0.0005, 'optimizer_name': 'Adam'}

    data_full_total = df_sample['NETINCOME'].values.reshape(-1, 1)
    data_len = len(data_full_total)
    
    # Preparación del dataset para Optuna (split 80/20)
    scaler_agg = StandardScaler()
    scaled_data_full = scaler_agg.fit_transform(data_full_total)
    train_val_size = int(data_len * 0.80)
    train_val_data = scaled_data_full[0:train_val_size, :]
    
    X_train_val_all, y_train_val_all = create_dataset(train_val_data, LOOK_BACK)
    train_split_point = int(len(X_train_val_all) * 0.80)

    X_train = X_train_val_all[:train_split_point]
    y_train = y_train_val_all[:train_split_point]
    X_val = X_train_val_all[train_split_point:]
    y_val = y_train_val_all[train_split_point:]

    if len(X_val) == 0:
        print("ADVERTENCIA: Datos insuficientes para validación. Usando entrenamiento/validación para Optuna.")
        X_train_final = X_train_val_all
        y_train_final = y_train_val_all
        # Optuna fallará sin val_loss. Se usa un split simple para asegurar la validación.
        X_train = X_train_val_all[:-5] 
        y_train = y_train_val_all[:-5]
        X_val = X_train_val_all[-5:]
        y_val = y_train_val_all[-5:]
        if len(X_val) == 0:
            print("ERROR: Fallo al crear set de validación para Optuna. Usando valores por defecto.")
            return {'n_units_l1': 256, 'n_units_l2': 128, 'dropout_rate': 0.3, 'learning_rate': 0.0005, 'optimizer_name': 'Adam'}
    
    # 6. Ejecución del Estudio de Optuna (Presupuesto)
    # 'sampler=optuna.samplers.TPESampler()' para Bayes; Default es TPE
    study = optuna.create_study(direction='minimize')
    study.optimize(
        lambda trial: objective(trial, X_train, y_train, X_val, y_val, LOOK_BACK), 
        n_trials=N_TRIALS_OPTUNA, 
        timeout=TIMEOUT_OPTUNA,
        show_progress_bar=True
    )
    
    # 7. Obtener y mostrar resultados
    print("\n[🏆] Búsqueda finalizada.")
    print(f"Mejor Valor de 'val_loss': {study.best_value:.6f}")
    print("Mejores Hiperparámetros encontrados:")
    
    best_params_table = [[k, v] for k, v in study.best_params.items()]
    print(tabulate(best_params_table, headers=["Hiperparámetro", "Valor Óptimo"], tablefmt="fancy_grid"))
    
    return study.best_params


# --------------------------------------------------------------------------------------
# --- REEMPLAZO: calculate_aggregated_metrics_comparison (Ahora usa los óptimos y los compara) ---
# --------------------------------------------------------------------------------------

# Se mantiene esta función, pero se elimina la comparación manual VAR1 y VAR2.
# En su lugar, se compara el modelo BASE (manual) con el modelo OPTUNA (mejorado)
def calculate_aggregated_metrics_comparison(data_full_total, portfolio_name, best_hparams):
    """
    Calcula las métricas de RMSE, MAE, R2 y DA para el portafolio agregado 
    comparando el modelo Base con el modelo optimizado por Optuna.
    """
    data_len = len(data_full_total)
    
    if data_len < MIN_DATA_POINTS:
        print("ADVERTENCIA: Datos insuficientes para métricas agregadas.")
        default_nan = (np.nan, np.nan, np.nan, np.nan)
        return {'BASE': default_nan, 'OPTUNA': default_nan}

    scaler_agg = StandardScaler()
    scaled_data_full = scaler_agg.fit_transform(data_full_total)

    train_size = int(data_len * 0.80)
    train_data = scaled_data_full[0:train_size, :]
    test_data = scaled_data_full[train_size - LOOK_BACK:, :]

    X_train, y_train = create_dataset(train_data, LOOK_BACK)
    X_test, y_test = create_dataset(test_data, LOOK_BACK)

    if len(X_test) == 0:
        default_nan = (np.nan, np.nan, np.nan, np.nan)
        return {'BASE': default_nan, 'OPTUNA': default_nan}

    X_train_reshaped = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))

    # --- ENTRENAMIENTO Y EVALUACIÓN DE MODELOS ---
    # 1. Modelo BASE (Arquitectura manual anterior)
    model_base = build_lstm_model(LOOK_BACK, 256, 128, 0.3, 0.0005, 'Adam') # Parámetros BASE manuales

    # 2. Modelo OPTUNA (Arquitectura con mejores hiperparámetros)
    model_optuna = build_lstm_model(
        LOOK_BACK, 
        best_hparams['n_units_l1'], 
        best_hparams['n_units_l2'], 
        best_hparams['dropout_rate'], 
        best_hparams['learning_rate'], 
        best_hparams['optimizer_name']
    )

    models = {
        'BASE': model_base,
        'OPTUNA': model_optuna
    }

    results = {}
    es_callback = EarlyStopping(monitor='loss', patience=PATIENCE_EARLY_STOPPING, verbose=0, restore_best_weights=True)

    for name, model in models.items():
        # Entrenar con el mismo split Training/Test
        model.fit(
            X_train_reshaped,
            y_train,
            epochs=EPOCHS,
            batch_size=1,
            verbose=0,
            callbacks=[es_callback]
        )
        
        rmse, mae, r2, da = calculate_metrics_for_model(model, X_test, y_test, scaler_agg)
        results[name] = (rmse, mae, r2, da)
        print(f"  > Resultados {name} (Agg Test): RMSE={rmse:,.2f}, MAE={mae:,.2f}, DA={da:.2%}")

    return results


# --------------------------------------------------------------------------------------
# --- REEMPLAZO: process_entity_for_portfolio (Usa los hiperparámetros óptimos) ---
# --------------------------------------------------------------------------------------

def process_entity_for_portfolio(entity_id, portfolio_id, portfolio_name, df_full, best_hparams):
    """
    Procesa, entrena, pronostica y GUARDA el modelo Y el SCALER para una ENTIDAD específica
    utilizando los **hiperparámetros óptimos** encontrados por Optuna.
    """
    df_entity = df_full[
        (df_full['PORTFOLIOID'] == entity_id)
    ].sort_values('DATE').reset_index(drop=True)

    if df_entity.empty:
        print(f"ADVERTENCIA: Entidad ID {entity_id} del Portafolio {portfolio_id} sin datos. Saltando.")
        return None, None, None, None, None

    data_full = df_entity['NETINCOME'].values.reshape(-1, 1)
    
    print(f"\n--- 📈 Procesando Entidad: ID {entity_id} (Portafolio {portfolio_name}) (Total de datos: {len(data_full)}) ---")

    if len(data_full) < MIN_DATA_POINTS:
        print(f"ADVERTENCIA: Entidad ID {entity_id} necesita al menos {MIN_DATA_POINTS} registros. Solo se tienen {len(data_full)}. Saltando.")
        return None, None, None, None, None

    # 3.2 ESCALADO DE DATOS
    scaler = StandardScaler()
    scaled_data_full = scaler.fit_transform(data_full)
    
    # === CORRECCIÓN CLAVE: GUARDAR EL SCALER ===
    scaler_filename = os.path.join(MODELS_DIR, f"scaler_entity_{entity_id}_p{portfolio_id}.joblib")
    joblib.dump(scaler, scaler_filename)
    print(f"[💾] Scaler guardado en: {scaler_filename}")
    # ==========================================
    
    # 3.3 PREPARACIÓN DEL SET DE ENTRENAMIENTO/VALIDACIÓN (80% del total)
    train_val_size = int(len(scaled_data_full) * 0.80)
    train_val_data = scaled_data_full[0:train_val_size, :]
    
    X_train_val_all, y_train_val_all = create_dataset(train_val_data, LOOK_BACK)
    
    if len(X_train_val_all) == 0:
        print(f"ADVERTENCIA: Datos insuficientes para crear dataset de entrenamiento/validación para la Entidad {entity_id}. Saltando.")
        return None, None, None, None, None

    # 3.4 SPLIT FIJO (80% Entrenamiento / 20% Validación del 80% inicial)
    train_split_point = int(len(X_train_val_all) * 0.80)

    X_train = X_train_val_all[:train_split_point]
    y_train = y_train_val_all[:train_split_point]
    X_val = X_train_val_all[train_split_point:]
    y_val = y_train_val_all[train_split_point:]
    
    # Determinar si hay datos de validación
    has_validation_data = len(X_val) > 0

    if not has_validation_data:
        print(f"ADVERTENCIA: Set de validación vacío. Usando el set completo como entrenamiento.")
        X_train_final = X_train_val_all
        y_train_final = y_train_val_all
        val_rmse, val_mae, val_r2 = np.nan, np.nan, np.nan
        monitor_metric = 'loss'
    else:
        X_train_final = X_train
        y_train_final = y_train
        monitor_metric = 'val_loss'

    X_train_reshaped = np.reshape(X_train_final, (X_train_final.shape[0], X_train_final.shape[1], 1))
    
    # 3.5 ENTRENAMIENTO FINAL CON HIPERPARÁMETROS ÓPTIMOS (Modelo OPTUNA)
    model_final = build_lstm_model(
        LOOK_BACK, 
        best_hparams['n_units_l1'], 
        best_hparams['n_units_l2'], 
        best_hparams['dropout_rate'], 
        best_hparams['learning_rate'], 
        best_hparams['optimizer_name']
    ) # Modelo OPTIMIZADO
    
    print(f"[✨] Entrenando con Optimizador: {best_hparams['optimizer_name']} | LR: {best_hparams['learning_rate']:.5f} | Drop: {best_hparams['dropout_rate']:.2f}")

    # Configuración de validation_data y EarlyStopping
    validation_tuple = (np.reshape(X_val, (X_val.shape[0], X_val.shape[1], 1)), y_val) if has_validation_data else None

    model_final.fit(
        X_train_reshaped, y_train_final,
        epochs=EPOCHS,
        batch_size=1,
        verbose=0,
        validation_data=validation_tuple,
        callbacks=[EarlyStopping(monitor=monitor_metric, patience=PATIENCE_EARLY_STOPPING, verbose=0, restore_best_weights=True)]
    )

    # 3.5.1 CÁLCULO DE MÉTRICAS DE VALIDACIÓN (Si aplica)
    if has_validation_data:
        X_val_reshaped = np.reshape(X_val, (X_val.shape[0], X_val.shape[1], 1))
        val_predict_scaled = model_final.predict(X_val_reshaped, verbose=0)
        val_predict_original = scaler.inverse_transform(val_predict_scaled)
        y_val_original = scaler.inverse_transform(y_val.reshape(-1, 1))

        val_rmse = np.sqrt(mean_squared_error(y_val_original, val_predict_original))
        val_mae = mean_absolute_error(y_val_original, val_predict_original)
        val_r2 = r2_score(y_val_original, val_predict_original)
        
        print(f"✅ Split Fijo 80/20 (Modelo OPTIMIZADO). RMSE Validación: {val_rmse:,.4f} | MAE Validación: {val_mae:,.4f} | R2 Validación: {val_r2:.4f}")
    else:
        print("ℹ️ Métrica de validación omitida. El modelo se entrenó en el set completo (80%).")

    
    model_filename = os.path.join(MODELS_DIR, f"lstm_entity_{entity_id}_p{portfolio_id}_optimized.keras") # Nuevo nombre de archivo
    model_final.save(model_filename)
    logging.info(f"Modelo guardado en: {model_filename}")
    print(f"[💾] Modelo guardado en: {model_filename}")
    
    # 3.6 PRONÓSTICO MULTI-PASO (usando el modelo optimizado)
    forecast = []
    last_sequence = scaled_data_full[-LOOK_BACK:]
    current_input = last_sequence.reshape(1, LOOK_BACK, 1)

    for i in range(FUTURE_MONTHS):
        next_prediction = model_final.predict(current_input, verbose=0)
        forecast.append(next_prediction[0, 0])
        new_sequence = np.append(current_input[:, 1:, :], next_prediction.reshape(1, 1, 1), axis=1)
        current_input = new_sequence

    forecast_original = scaler.inverse_transform(np.array(forecast).reshape(-1, 1)).flatten()
    
    return forecast_original, len(data_full), val_rmse, val_mae, val_r2


def process_portfolio_by_entity(portfolio_id, df_full, all_portfolio_results, best_hparams):
    """
    Función principal para el portafolio: Acumula pronósticos, calcula métricas (comparativas) y almacena resultados.
    """
    df_portfolio_group = df_full[df_full['PORTFOLIOID'] == portfolio_id].copy()
    if df_portfolio_group.empty: return

    portfolio_name = df_portfolio_group['PORTFOLIONAME'].iloc[0]
    # Se asume que PORTFOLIOID y ENTITYID son lo mismo.
    entity_ids = df_portfolio_group['PORTFOLIOID'].unique()
    
    print("\n" + "#" * 80)
    print(f"### 🎯 INICIANDO ANÁLISIS ACUMULADO PARA PORTAFOLIO: ID {portfolio_id} - {portfolio_name}")
    print("#" * 80)

    total_forecast = np.zeros(FUTURE_MONTHS)
    successful_entities = 0
    # Estas listas ahora guardarán las métricas del SPLIT FIJO del modelo OPTIMIZADO
    all_rmse_val = []
    all_mae_val = []
    all_r2_val = []

    for e_id in entity_ids:
        # El pronóstico y las métricas VAL usan el Modelo OPTIMIZADO
        entity_forecast, data_points, rmse_val, mae_val, r2_val = process_entity_for_portfolio(e_id, portfolio_id, portfolio_name, df_full, best_hparams)
        
        if entity_forecast is not None:
            total_forecast += entity_forecast
            successful_entities += 1
            if not np.isnan(rmse_val):
                all_rmse_val.append(rmse_val)
                all_mae_val.append(mae_val)
                all_r2_val.append(r2_val)
            
    if successful_entities == 0:
        print(f"⚠️ Portafolio {portfolio_id} saltado: Ninguna entidad cumplió con los requisitos mínimos de datos.")
        return

    df_portfolio_total = df_portfolio_group.groupby('DATE')['NETINCOME'].sum().reset_index()
    data_full_total = df_portfolio_total['NETINCOME'].values.reshape(-1, 1)
    
    print("\n--- 📊 Calculando Métricas para la serie de tiempo TOTAL del Portafolio (Comparación) ---")
    
    # 💥 Llamada a la nueva función que entrena y compara el modelo BASE vs OPTUNA
    agg_results = calculate_aggregated_metrics_comparison(data_full_total, portfolio_name, best_hparams)

    # Desempaquetar resultados de cada modelo
    rmse_agg, mae_agg, r2_agg, da_agg = agg_results['BASE']
    rmse_agg_optuna, mae_agg_optuna, r2_agg_optuna, da_agg_optuna = agg_results['OPTUNA']
    
    # Promedios de las métricas de Validación Fija (del modelo OPTIMIZADO)
    rmse_val_avg_report = np.mean(all_rmse_val) if all_rmse_val else np.nan
    mae_val_avg_report = np.mean(all_mae_val) if all_mae_val else np.nan
    r2_val_avg_report = np.mean(all_r2_val) if all_r2_val else np.nan

    all_portfolio_results.append({
        'PORTFOLIOID': portfolio_id,
        'PORTFOLIONAME': portfolio_name,
        'ENTITY_ID_FOR_TEST': entity_ids[0],
        # Métricas de Validación Fija (antes CV) - Modelo OPTIMIZADO
        'RMSE_VAL_AVG': rmse_val_avg_report,
        'MAE_VAL_AVG': mae_val_avg_report,
        'R2_VAL_AVG': r2_val_avg_report,
        
        # Métricas de Agregación (Modelo Base)
        'RMSE_TEST_AGG': rmse_agg,
        'MAE_TEST_AGG': mae_agg,
        'R2_TEST_AGG': r2_agg,
        'DA_TEST_AGG': da_agg,
        
        # Métricas de Agregación (Modelo Optuna)
        'RMSE_TEST_AGG_OPTUNA': rmse_agg_optuna,
        'MAE_TEST_AGG_OPTUNA': mae_agg_optuna,
        'DA_TEST_AGG_OPTUNA': da_agg_optuna,
        
        'DATA_POINTS': len(data_full_total),
        'FORECAST_MONTHS': FUTURE_MONTHS,
        'FORECAST': total_forecast
    })
    
    print(f"\nPortafolio {portfolio_id} - {portfolio_name} | NETINCOME Proyectado:")
    for i, value in enumerate(total_forecast):
        print(f"Mes {i+1} (Acumulado): {value:,.2f}")

    # GRÁFICO 
    plt.figure(figsize=(18, 8))
    time_axis = df_portfolio_total['DATE']
    plt.plot(time_axis, data_full_total, label='NETINCOME Total Real del Portafolio', color='blue', linewidth=2)
    last_date = time_axis.iloc[-1]
    future_dates = pd.date_range(start=last_date, periods=FUTURE_MONTHS + 1, freq='MS')[1:]
    plt.plot(future_dates, total_forecast, label=f'Pronóstico ACUMULADO OPTIMIZADO ({successful_entities} Entidades)', color='red', linestyle='-', marker='o', linewidth=2)
    plt.title(f'Pronóstico ACUMULADO de NETINCOME para el Portafolio: {portfolio_name} (ID: {portfolio_id})')
    plt.xlabel('Fecha')
    plt.ylabel('NETINCOME Acumulado')
    plt.legend()
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plot_save_path = os.path.join(OUTPUT_DIR, f"forecast_portfolio_{portfolio_id}_accumulated_optimized.png")
    plt.savefig(plot_save_path)
    plt.close()
    print(f"Gráfico ACUMULADO guardado en: {plot_save_path}")
    print("-" * 50)

# --------------------------------------------------------------------------------------
# --- FUNCIÓN DE PRUEBA DE MODELO GUARDADO (SE ADAPTA AL NUEVO NOMBRE) ---
# --------------------------------------------------------------------------------------

def test_saved_model_prediction(entity_id, portfolio_id, look_back):
    """
    Carga el modelo (OPTIMIZADO) y el scaler guardados, predice con datos dummy y desescala el resultado.
    """
    model_filename = os.path.join(MODELS_DIR, f"lstm_entity_{entity_id}_p{portfolio_id}_optimized.keras")
    scaler_filename = os.path.join(MODELS_DIR, f"scaler_entity_{entity_id}_p{portfolio_id}.joblib")

    if not os.path.exists(model_filename) or not os.path.exists(scaler_filename):
        print(f"\n⚠️ ARCHIVO(S) NO ENCONTRADO: Modelo o Scaler para '{entity_id}' no existe. Saltando prueba.")
        return
    
    try:
        # 1. Cargar el modelo y el scaler
        model_loaded = load_model(model_filename)
        scaler_loaded = joblib.load(scaler_filename) # Cargar el scaler
        
        # 2. Generar datos de entrada aleatorios (DUMMY DATA - Escalados)
        NUM_TEST_SEQUENCES = 5
        dummy_input = np.random.randn(NUM_TEST_SEQUENCES, look_back, 1)

        print(f"\n[🔬] Prueba de Carga del Modelo: {model_filename}")
        
        # 3. Realizar la predicción (Salida ESCALADA)
        dummy_predictions_scaled = model_loaded.predict(dummy_input, verbose=0)
        
        # 4. Desescalar la predicción (Salida REAL)
        dummy_predictions_real = scaler_loaded.inverse_transform(dummy_predictions_scaled.reshape(-1, 1))

        # 5. Mostrar el resultado REAL
        test_data_table = []
        for i in range(NUM_TEST_SEQUENCES):
            
            # Desescalamos el primer valor de la secuencia dummy para mostrar un valor de entrada 'real' simulado
            dummy_input_unscaled = scaler_loaded.inverse_transform(dummy_input[i, 0, 0].reshape(-1, 1))[0, 0]
            
            output_val_real = f"{dummy_predictions_real[i, 0]:,.2f}"
            input_val_real = f"{dummy_input_unscaled:,.2f}"

            test_data_table.append([i + 1, input_val_real, output_val_real])

            # === IMPRESIÓN CORREGIDA: VALORES REALES ===
        print("\n    RESULTADOS DE PRUEBA CON DATOS  (Valores Monetarios Reales):")
        print(tabulate(
            test_data_table,
            headers=["Secuencia #", "Primer valor de entrada (Simulado)", "Predicción de Salida (NETINCOME)"],
            tablefmt="fancy_grid"
        ))
        print("    ✅ El modelo .keras genera y desescala predicciones correctamente.")
        # ==========================================

    except Exception as e:
        print(f"\n❌ ERROR FATAL EN PRUEBA DEL MODELO {model_filename}: {e}")
        logging.error(f"Error al probar el modelo {model_filename}: {e}")



In [28]:
# =========================================================
# === 3. EJECUCIÓN DEL PIPELINE Y REPORTE FINAL ===
# =========================================================

# === NUEVO PASO: BÚSQUEDA DE HIPERPARÁMETROS ===
BEST_HYPERPARAMS = find_optimal_hyperparameters(df_full)

# === AHORA SE USAN LOS HIPERPARÁMETROS ÓPTIMOS EN EL LOOP ===
logging.info("4. EJECUCIÓN DEL ANÁLISIS PARA TODOS LOS PORTAFOLIOS CON HIPERPARÁMETROS ÓPTIMOS")
for p_id in portfolio_ids:
    process_portfolio_by_entity(p_id, df_full, all_portfolio_results, BEST_HYPERPARAMS)

print("\n🎉 Análisis, pronóstico y ACUMULACIÓN completado. Revisa la carpeta './REPORTS'.")

logging.info("5. GENERACIÓN DEL RESUMEN FINAL Y EXPORTACIÓN A EXCEL")

if all_portfolio_results:
    # 5.1 PREPARAR DATOS Y EXPORTAR A EXCEL (ADAPTADO)
    summary_data_list = []
    forecast_headers = [f'Pronóst.Mes {i+1}' for i in range(FUTURE_MONTHS)]

    for result in all_portfolio_results:
        row = {
            'Portafolio': result['PORTFOLIONAME'],
            '# Puntos': result['DATA_POINTS'],
            # Métricas de Validación Fija (Modelo OPTIMIZADO)
            'MAE(Val Opt)': result['MAE_VAL_AVG'],
            'RMSE(Val Opt)': result['RMSE_VAL_AVG'],
            # Base (Agregado)
            'MAE(Agg P Base)': result['MAE_TEST_AGG'],
            'RMSE(Agg P Base)': result['RMSE_TEST_AGG'],
            'DA(Agg P Base)': result['DA_TEST_AGG'],
            # OPTUNA (Agregado)
            'MAE(Agg P Opt)': result['MAE_TEST_AGG_OPTUNA'],
            'RMSE(Agg P Opt)': result['RMSE_TEST_AGG_OPTUNA'],
            'DA(Agg P Opt)': result['DA_TEST_AGG_OPTUNA'],
        }
        for i, val in enumerate(result['FORECAST']):
            row[forecast_headers[i]] = val
        summary_data_list.append(row)

    df_summary = pd.DataFrame(summary_data_list)
    excel_filename = os.path.join(OUTPUT_DIR, f"SUMMARY_RESULTS_PORTFOLIO_LSTM_OPTIMIZED_COMPARISON_EPOCH{EPOCHS}.xlsx")
    
    try:
        df_summary.to_excel(excel_filename, index=False, sheet_name='REPORT')
        print(f"\n[📄] **Archivo Excel con el resumen guardado exitosamente en:** {excel_filename}")
    except Exception as e:
        print(f"\n⚠️ ERROR al intentar guardar el archivo Excel: {e}")

    # 5.2.1 IMPRIMIR COMPARACIÓN ARQUITECTÓNICA DE MODELOS (ADAPTADO)
    
    arch_comparison_data = [
        ["Capa de Entrada", "Input(shape=(12, 1))", "Input(shape=(12, 1))"],
        ["LSTM 1 (Unidades)", "256 (Fijo)", f"{BEST_HYPERPARAMS['n_units_l1']} (Óptimo)"],
        ["Dropout 1 (Tasa)", "0.3 (Fijo)", f"{BEST_HYPERPARAMS['dropout_rate']:.2f} (Óptimo)"],
        ["LSTM 2 (Unidades)", "128 (Fijo)", f"{BEST_HYPERPARAMS['n_units_l2']} (Óptimo)"],
        ["Dropout 2 (Tasa)", "0.3 (Fijo)", f"{BEST_HYPERPARAMS['dropout_rate']:.2f} (Óptimo)"],
        ["Capa Densa (Salida)", "1 (lineal)", "1 (lineal)"],
        ["**OPTIMIZADOR**", "**Adam** (Fijo)", f"**{BEST_HYPERPARAMS['optimizer_name']}** (Óptimo)"],
        ["**TASA APRENDIZAJE**", "**0.0005** (Fijo)", f"**{BEST_HYPERPARAMS['learning_rate']:.5f}** (Óptimo)"],
    ]
    
    headers_arch = ["Característica", "Modelo BASE (Manual)", "Modelo OPTUNA (Optimizado)"]
    
    print("\n" + "="*100)
    print("      CUADRO COMPARATIVO DE ARQUITECTURAS LSTM (BASE vs. OPTUNA)")
    print("="*100)
    print(tabulate(arch_comparison_data, headers=headers_arch, tablefmt="fancy_grid"))
    print("\n" + "-"*100)

    # 5.2 IMPRIMIR RESUMEN EN CONSOLA (TABULATE) - CON COMPARACIÓN
    
    headers_metrics = [
        "MAE(Val Opt)", "RMSE(Val Opt)",
        "RMSE(Agg Base)", "DA(Agg Base)",
        "RMSE(Agg Optuna)", "DA(Agg Optuna)",
    ]
    headers = ["Portafolio", "# Puntos"] + headers_metrics + forecast_headers
    
    tabulate_data = []
    for result in all_portfolio_results:
        # Métricas de Validación Fija (modelo individual - OPTIMIZADO)
        mae_val_val = f"{result['MAE_VAL_AVG']:,.2f}" if not np.isnan(result['MAE_VAL_AVG']) else "N/A"
        rmse_val_val = f"{result['RMSE_VAL_AVG']:,.2f}" if not np.isnan(result['RMSE_VAL_AVG']) else "N/A"
        
        # Métricas del modelo BASE (Agregado)
        rmse_agg_val = f"{result['RMSE_TEST_AGG']:,.2f}" if not np.isnan(result['RMSE_TEST_AGG']) else "Insuficiente"
        da_agg_val = f"{result['DA_TEST_AGG']:.2%}" if not np.isnan(result['DA_TEST_AGG']) else "Insuficiente"
        
        # Métricas del modelo OPTUNA (Agregado)
        rmse_agg_optuna_val = f"{result['RMSE_TEST_AGG_OPTUNA']:,.2f}" if not np.isnan(result['RMSE_TEST_AGG_OPTUNA']) else "Insuficiente"
        da_agg_optuna_val = f"{result['DA_TEST_AGG_OPTUNA']:.2%}" if not np.isnan(result['DA_TEST_AGG_OPTUNA']) else "Insuficiente"
        
        row = [
            result['PORTFOLIONAME'],
            f"{result['DATA_POINTS']}",
            mae_val_val,
            rmse_val_val,
            rmse_agg_val, da_agg_val,
            rmse_agg_optuna_val, da_agg_optuna_val,
        ]
        row.extend([f"{val:,.2f}" for val in result['FORECAST']])
        tabulate_data.append(row)

    print("\n" + "="*210)
    print("                      RESUMEN DE RESULTADOS DE PRONÓSTICO LSTM (NETINCOME) ACUMULADO Y COMPARACIÓN DE OPTIMIZACIÓN")
    print("                      Métricas (Val Opt) = Validación Fija (80/20 del 80%) de la entidad individual (Modelo Optimizado)")
    print("                      Métricas (Agg P) = Prueba en la serie de tiempo Total del Portafolio (split 80/20)")
    print("                      DA = Directional Accuracy ('Accuracy' para regresión)")
    print("="*210)
    print(tabulate(tabulate_data, headers=headers, tablefmt="fancy_grid"))

    # 6. EJECUCIÓN DE LA RUTINA DE PRUEBA FINAL
    print("\n" + "="*80)
    print("=== 🔬 INICIO DE LA RUTINA DE PRUEBA DE MODELOS GUARDADOS ===")
    print("==================================================================================")
    
    # Se usa el último resultado para la prueba, asumiendo que el modelo y scaler se guardaron correctamente
    last_result = all_portfolio_results[-1]
    
    test_saved_model_prediction(
        entity_id=last_result['ENTITY_ID_FOR_TEST'],
        portfolio_id=last_result['PORTFOLIOID'],
        look_back=LOOK_BACK
    )
    print("-" * 80)
    logging.info("FINALIZA ENTRENAMIENTO Y PRUEBA")

else:
    print("\n⚠️ No se pudieron generar resultados para el resumen final.")

[I 2025-10-30 00:18:11,840] A new study created in memory with name: no-name-b0caae93-c9ad-42cf-bb5d-8145d0e5d327



=== ⚙️ INICIO BÚSQUEDA DE HIPERPARÁMETROS (OPTUNA: Random Search/TPE) ===
Presupuesto: 15 pruebas o 60.0 minutos.


  0%|          | 0/15 [00:00<?, ?it/s]

[I 2025-10-30 00:21:38,139] Trial 0 finished with value: 0.24662618339061737 and parameters: {'n_units_l1': 64, 'n_units_l2': 32, 'dropout_rate': 0.3918467314074989, 'learning_rate': 2.4885567141401323e-05, 'optimizer_name': 'RMSprop'}. Best is trial 0 with value: 0.24662618339061737.
[I 2025-10-30 00:21:47,934] Trial 1 finished with value: 0.23114442825317383 and parameters: {'n_units_l1': 64, 'n_units_l2': 64, 'dropout_rate': 0.35823646071623716, 'learning_rate': 0.0008961272805768954, 'optimizer_name': 'Adam'}. Best is trial 1 with value: 0.23114442825317383.
[I 2025-10-30 00:22:41,655] Trial 2 finished with value: 0.22934481501579285 and parameters: {'n_units_l1': 256, 'n_units_l2': 64, 'dropout_rate': 0.2605190524930088, 'learning_rate': 5.724336793922931e-05, 'optimizer_name': 'RMSprop'}. Best is trial 2 with value: 0.22934481501579285.
[I 2025-10-30 00:22:56,001] Trial 3 finished with value: 0.24960267543792725 and parameters: {'n_units_l1': 128, 'n_units_l2': 32, 'dropout_rate'